In [ ]:
!pip -q install kaggle numpy pandas scikit-learn tensorflow joblib dill lime matplotlib


In [ ]:
from google.colab import files
from pathlib import Path
import os
import shutil

kaggle_dir = Path('/root/.kaggle')
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_json = kaggle_dir / 'kaggle.json'

if not kaggle_json.exists():
    print('Upload kaggle.json from your machine')
    uploaded = files.upload()
    if 'kaggle.json' not in uploaded:
        raise FileNotFoundError('You must upload kaggle.json')
    if Path('kaggle.json').exists():
        shutil.move('kaggle.json', kaggle_json)

os.chmod(kaggle_json, 0o600)
print('Using Kaggle credentials at', kaggle_json)


In [ ]:
DATASET_SLUG = 'remarezirafsanjani/cse-cic-ids2018-cleaned-and-match-cicflowmeter'
BENIGN_LABEL = 'Benign'
LABEL_COLUMN_OVERRIDE = ''
DOWNLOAD_DIR = '/content/kaggle_data'
OUTPUT_DIR = '/content/models_out_v3'

MAX_BENIGN_ROWS_TOTAL = 25000
MAX_ATTACK_ROWS_PER_CLASS = 8000
CHUNK_SIZE = 30000
RANDOM_STATE = 42
TEST_SIZE = 0.2

CLASSIFIER_MIN_ROWS_PER_CLASS = 3000
CLASSIFIER_MAX_ROWS_PER_CLASS = 12000
RF_TUNE_SAMPLE_MAX_ROWS = 30000
RF_SEARCH_ITER = 6
RF_CV_SPLITS = 3

ENABLE_GPU_MIXED_PRECISION = True
AE_EPOCHS = 40
AE_BATCH_SIZE = 256
AE_MAX_BENIGN_TRAIN_ROWS = 22000
AE_MAX_ATTACK_CALIB_ROWS = 6000
AE_CLIP_LOWER_QUANTILE = 0.01
AE_CLIP_UPPER_QUANTILE = 0.99
AE_KEEP_BENIGN_ROW_QUANTILE = 0.98
LIME_MAX_TRAIN_ROWS = 2500

print('DATASET_SLUG =', DATASET_SLUG)
print('BENIGN_LABEL =', BENIGN_LABEL)
print('DOWNLOAD_DIR =', DOWNLOAD_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)
print('MAX_BENIGN_ROWS_TOTAL =', MAX_BENIGN_ROWS_TOTAL)
print('MAX_ATTACK_ROWS_PER_CLASS =', MAX_ATTACK_ROWS_PER_CLASS)
print('CHUNK_SIZE =', CHUNK_SIZE)
print('CLASSIFIER_MIN_ROWS_PER_CLASS =', CLASSIFIER_MIN_ROWS_PER_CLASS)
print('CLASSIFIER_MAX_ROWS_PER_CLASS =', CLASSIFIER_MAX_ROWS_PER_CLASS)
print('RF_TUNE_SAMPLE_MAX_ROWS =', RF_TUNE_SAMPLE_MAX_ROWS)
print('RF_SEARCH_ITER =', RF_SEARCH_ITER)
print('ENABLE_GPU_MIXED_PRECISION =', ENABLE_GPU_MIXED_PRECISION)


In [ ]:
import gc
import glob
import json
import os
import pickle
import re
import shutil
import subprocess
import time
from collections import Counter
from pathlib import Path

import dill
import joblib
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lime import lime_tabular
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import mixed_precision

APP_FEATURES = [
    'FlowDuration', 'BwdPacketLenMax', 'BwdPacketLenMin', 'BwdPacketLenMean', 'BwdPacketLenStd',
    'FlowIATMean', 'FlowIATStd', 'FlowIATMax', 'FlowIATMin', 'FwdIATTotal', 'FwdIATMean',
    'FwdIATStd', 'FwdIATMax', 'FwdIATMin', 'BwdIATTotal', 'BwdIATMean', 'BwdIATStd',
    'BwdIATMax', 'BwdIATMin', 'FwdPSHFlags', 'FwdPackets_s', 'MaxPacketLen', 'PacketLenMean',
    'PacketLenStd', 'PacketLenVar', 'FINFlagCount', 'SYNFlagCount', 'PSHFlagCount',
    'ACKFlagCount', 'URGFlagCount', 'AvgPacketSize', 'AvgBwdSegmentSize', 'InitWinBytesFwd',
    'InitWinBytesBwd', 'ActiveMin', 'IdleMean', 'IdleStd', 'IdleMax', 'IdleMin'
]

FEATURE_ALIASES = {
    'FlowDuration': ['Flow Duration'],
    'BwdPacketLenMax': ['Bwd Packet Length Max', 'BwdPacketLengthMax', 'Bwd Pkt Len Max'],
    'BwdPacketLenMin': ['Bwd Packet Length Min', 'BwdPacketLengthMin', 'Bwd Pkt Len Min'],
    'BwdPacketLenMean': ['Bwd Packet Length Mean', 'BwdPacketLengthMean', 'Bwd Pkt Len Mean'],
    'BwdPacketLenStd': ['Bwd Packet Length Std', 'BwdPacketLengthStd', 'Bwd Pkt Len Std'],
    'FlowIATMean': ['Flow IAT Mean'],
    'FlowIATStd': ['Flow IAT Std'],
    'FlowIATMax': ['Flow IAT Max'],
    'FlowIATMin': ['Flow IAT Min'],
    'FwdIATTotal': ['Fwd IAT Total', 'Fwd IAT Tot'],
    'FwdIATMean': ['Fwd IAT Mean'],
    'FwdIATStd': ['Fwd IAT Std'],
    'FwdIATMax': ['Fwd IAT Max'],
    'FwdIATMin': ['Fwd IAT Min'],
    'BwdIATTotal': ['Bwd IAT Total', 'Bwd IAT Tot'],
    'BwdIATMean': ['Bwd IAT Mean'],
    'BwdIATStd': ['Bwd IAT Std'],
    'BwdIATMax': ['Bwd IAT Max'],
    'BwdIATMin': ['Bwd IAT Min'],
    'FwdPSHFlags': ['Fwd PSH Flags'],
    'FwdPackets_s': ['Fwd Packets/s', 'FwdPackets/s', 'Fwd Pkts/s'],
    'MaxPacketLen': ['Max Packet Length', 'Packet Length Max', 'PacketLengthMax', 'Pkt Len Max'],
    'PacketLenMean': ['Packet Length Mean', 'PacketLengthMean', 'Pkt Len Mean'],
    'PacketLenStd': ['Packet Length Std', 'PacketLengthStd', 'Pkt Len Std'],
    'PacketLenVar': ['Packet Length Variance', 'PacketLengthVariance', 'Pkt Len Var'],
    'FINFlagCount': ['FIN Flag Count', 'FIN Flag Cnt'],
    'SYNFlagCount': ['SYN Flag Count', 'SYN Flag Cnt'],
    'PSHFlagCount': ['PSH Flag Count', 'PSH Flag Cnt'],
    'ACKFlagCount': ['ACK Flag Count', 'ACK Flag Cnt'],
    'URGFlagCount': ['URG Flag Count', 'URG Flag Cnt'],
    'AvgPacketSize': ['Average Packet Size', 'AveragePacketSize', 'Pkt Size Avg'],
    'AvgBwdSegmentSize': ['Avg Bwd Segment Size', 'Bwd Segment Size Avg', 'BwdSegmentSizeAvg', 'Bwd Seg Size Avg'],
    'InitWinBytesFwd': ['Init_Win_bytes_forward', 'Fwd Init Win Bytes', 'FWDInitWinBytes', 'FwdInitWinBytes', 'Init Fwd Win Byts'],
    'InitWinBytesBwd': ['Init_Win_bytes_backward', 'Bwd Init Win Bytes', 'BwdInitWinBytes', 'Init Bwd Win Byts'],
    'ActiveMin': ['Active Min'],
    'IdleMean': ['Idle Mean'],
    'IdleStd': ['Idle Std'],
    'IdleMax': ['Idle Max'],
    'IdleMin': ['Idle Min'],
}

LABEL_ALIASES = ['Classification', 'Label', 'label', 'Class', 'class']

TOKEN_NORMALIZATION = {
    'forward': 'fwd',
    'backward': 'bwd',
    'packet': 'pkt',
    'packets': 'pkt',
    'pkts': 'pkt',
    'length': 'len',
    'total': 'tot',
    'count': 'cnt',
    'counts': 'cnt',
    'average': 'avg',
    'variance': 'var',
    'segment': 'seg',
    'bytes': 'byts',
    'byte': 'byts',
}

rng = np.random.default_rng(RANDOM_STATE)

def setup_tensorflow_runtime(enable_mixed_precision=True):
    gpu_devices = tf.config.list_physical_devices('GPU')
    for gpu in gpu_devices:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception:
            pass

    if gpu_devices and enable_mixed_precision:
        try:
            mixed_precision.set_global_policy('mixed_float16')
        except Exception as exc:
            print('Mixed precision disabled:', exc)
            mixed_precision.set_global_policy('float32')
    else:
        mixed_precision.set_global_policy('float32')

    status = {
        'gpu_count': len(gpu_devices),
        'gpu_names': [device.name for device in gpu_devices],
        'mixed_precision_policy': mixed_precision.global_policy().name,
    }
    print('TensorFlow GPU count:', status['gpu_count'])
    print('Mixed precision policy:', status['mixed_precision_policy'])
    if status['gpu_names']:
        print('GPU devices:', status['gpu_names'])
    return status

GPU_STATUS = setup_tensorflow_runtime(ENABLE_GPU_MIXED_PRECISION)

def make_rf_tuning_subset(X_train, y_train, max_rows, random_state=42):
    if max_rows is None or len(X_train) <= max_rows:
        return X_train, y_train

    class_counts = y_train.value_counts()
    stratify_labels = y_train if (class_counts >= 2).all() else None
    try:
        X_tune, _, y_tune, _ = train_test_split(
            X_train,
            y_train,
            train_size=max_rows,
            random_state=random_state,
            stratify=stratify_labels,
        )
    except ValueError:
        sampled_idx = y_train.sample(n=max_rows, random_state=random_state).index
        X_tune = X_train.loc[sampled_idx]
        y_tune = y_train.loc[sampled_idx]

    return X_tune, y_tune

def safe_cv_splits(y_values, requested_splits):
    if len(y_values) == 0:
        return 0
    min_class_count = int(y_values.value_counts().min())
    return min(requested_splits, min_class_count)

def oversample_classifier_training_data(X_train, y_train, min_rows_per_class, max_rows_per_class, random_state=42):
    x_parts = [X_train]
    y_parts = [y_train]
    counts_before = y_train.value_counts().sort_index()
    for label, current_count in counts_before.items():
        target_count = min(max_rows_per_class, max(min_rows_per_class, int(current_count)))
        if current_count >= target_count:
            continue
        extra_needed = target_count - int(current_count)
        label_idx = y_train[y_train == label].index
        sampled_idx = y_train.loc[label_idx].sample(n=extra_needed, replace=True, random_state=random_state).index
        x_parts.append(X_train.loc[sampled_idx].copy())
        y_parts.append(y_train.loc[sampled_idx].copy())

    X_aug = pd.concat(x_parts, ignore_index=True, copy=False)
    y_aug = pd.concat(y_parts, ignore_index=True, copy=False)
    counts_after = y_aug.value_counts().sort_index()
    return X_aug, y_aug, counts_before, counts_after

def normalize_name(name):
    return ''.join(ch.lower() for ch in str(name) if ch.isalnum())

def split_tokens(name):
    text = re.sub(r'([a-z])([A-Z])', r'\1 \2', str(name))
    text = re.sub(r'[^A-Za-z0-9]+', ' ', text)
    raw_tokens = [token.lower() for token in text.split() if token]
    return [TOKEN_NORMALIZATION.get(token, token) for token in raw_tokens]

def token_signature(name):
    return tuple(sorted(split_tokens(name)))

def find_best_column_match(columns, candidates):
    exact_map = {normalize_name(col): col for col in columns}
    signature_map = {}
    for col in columns:
        signature_map.setdefault(token_signature(col), col)

    for candidate in candidates:
        hit = exact_map.get(normalize_name(candidate))
        if hit:
            return hit

    for candidate in candidates:
        hit = signature_map.get(token_signature(candidate))
        if hit:
            return hit

    return None

def dataframe_mb(df):
    return float(df.memory_usage(deep=True).sum() / (1024 ** 2))

def class_cap(label):
    return MAX_BENIGN_ROWS_TOTAL if str(label).strip() == BENIGN_LABEL else MAX_ATTACK_ROWS_PER_CLASS

def has_csv_files(input_dir):
    return bool(glob.glob(os.path.join(input_dir, '**', '*.csv'), recursive=True))

def download_kaggle_dataset(dataset_slug, download_dir):
    os.makedirs(download_dir, exist_ok=True)
    if has_csv_files(download_dir):
        print('CSV files already exist in', download_dir, '- skip download')
        return
    completed = subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', dataset_slug, '-p', download_dir, '--unzip'],
        check=False,
        capture_output=True,
        text=True,
    )
    if completed.returncode != 0:
        stderr = (completed.stderr or '').strip()
        stdout = (completed.stdout or '').strip()
        details = stderr or stdout or 'Kaggle CLI failed without output'
        raise RuntimeError(f'Could not download dataset {dataset_slug}: {details}')
    print('Downloaded dataset to', download_dir)

def resolve_required_columns(columns, label_override=''):
    resolved = {}
    for feature in APP_FEATURES:
        candidates = [feature] + FEATURE_ALIASES.get(feature, [])
        found = find_best_column_match(columns, candidates)
        if not found:
            raise KeyError(f"Missing required feature column for '{feature}'. Checked aliases: {candidates}")
        resolved[feature] = found

    if label_override:
        if label_override not in columns:
            raise KeyError(f"Could not find label column '{label_override}'")
        label_col = label_override
    else:
        label_col = find_best_column_match(columns, LABEL_ALIASES)
        if label_col is None:
            raise KeyError(f'Could not find label column. Tried: {LABEL_ALIASES}')
    return resolved, label_col

def clean_chunk(chunk, label_col):
    numeric_cols = [col for col in chunk.columns if col != label_col]
    chunk[numeric_cols] = chunk[numeric_cols].apply(pd.to_numeric, errors='coerce').astype('float32')
    chunk[label_col] = chunk[label_col].astype(str).str.strip()
    chunk = chunk.replace([np.inf, -np.inf], np.nan)
    return chunk

def load_balanced_training_data(input_dir, label_override='', chunk_size=30000):
    pattern = os.path.join(input_dir, '**', '*.csv')
    x_parts = []
    y_parts = []
    kept_counts = Counter()
    detected_label_col = None

    for path in sorted(glob.glob(pattern, recursive=True)):
        try:
            header = pd.read_csv(path, nrows=0)
            resolved_features, label_col = resolve_required_columns(header.columns.tolist(), label_override)
            usecols = list(dict.fromkeys(list(resolved_features.values()) + [label_col]))

            kept_before = sum(kept_counts.values())
            for chunk in pd.read_csv(path, usecols=usecols, chunksize=chunk_size, low_memory=False):
                chunk = clean_chunk(chunk, label_col)
                selected = chunk[[resolved_features[f] for f in APP_FEATURES]].copy()
                selected.columns = APP_FEATURES
                labels = chunk[label_col].astype(str).str.strip()

                for current_label in labels.unique().tolist():
                    current_mask = labels == current_label
                    remaining = class_cap(current_label) - kept_counts[current_label]
                    if remaining <= 0:
                        continue

                    label_idx = labels[current_mask].index.to_numpy()
                    if len(label_idx) == 0:
                        continue

                    take_n = min(len(label_idx), remaining)
                    if take_n < len(label_idx):
                        chosen_idx = rng.choice(label_idx, size=take_n, replace=False)
                    else:
                        chosen_idx = label_idx

                    part_x = selected.loc[chosen_idx].copy()
                    part_y = labels.loc[chosen_idx].copy()
                    x_parts.append(part_x)
                    y_parts.append(part_y)
                    kept_counts[current_label] += len(part_x)

                del chunk, selected, labels
                gc.collect()

            kept_after = sum(kept_counts.values())
            print(f'Processed: {path} kept_rows_added={kept_after - kept_before}')
            detected_label_col = label_col
            del header
            gc.collect()
        except Exception as exc:
            print('Skip:', path, exc)

    if not x_parts:
        raise ValueError('Could not find any CSV file for training')

    X = pd.concat(x_parts, ignore_index=True, copy=False).astype('float32')
    y = pd.concat(y_parts, ignore_index=True, copy=False).astype(str).str.strip()

    del x_parts, y_parts
    gc.collect()

    class_counts = y.value_counts()
    rare_labels = class_counts[class_counts < 2].index.tolist()
    if rare_labels:
        mask = ~y.isin(rare_labels)
        X = X.loc[mask].reset_index(drop=True)
        y = y.loc[mask].reset_index(drop=True)
        print('Dropped labels with <2 rows:', rare_labels)

    print('Final balanced class counts:')
    print(y.value_counts())
    return X, y, detected_label_col

def compute_clip_bounds(frame, lower_q=0.01, upper_q=0.99):
    lower = frame.quantile(lower_q).astype('float32')
    upper = frame.quantile(upper_q).astype('float32')
    return lower, upper

def clip_frame(frame, lower_bounds, upper_bounds):
    return frame.clip(lower=lower_bounds, upper=upper_bounds, axis=1)

def filter_benign_rows_for_autoencoder(frame, lower_bounds, upper_bounds, keep_quantile=0.98):
    clipped = clip_frame(frame, lower_bounds, upper_bounds)
    row_shift = (frame - clipped).abs().sum(axis=1)
    threshold = float(row_shift.quantile(keep_quantile))
    keep_mask = row_shift <= threshold
    filtered = frame.loc[keep_mask].copy()
    metrics = {
        'input_rows': int(len(frame)),
        'kept_rows': int(keep_mask.sum()),
        'dropped_rows': int((~keep_mask).sum()),
        'keep_quantile': float(keep_quantile),
        'row_shift_threshold': threshold,
    }
    return filtered, metrics

def choose_autoencoder_threshold(benign_scores, attack_scores):
    if len(attack_scores) == 0:
        threshold = float(np.quantile(benign_scores, 0.995))
        return threshold, {'precision': None, 'recall': None, 'f1': None, 'strategy': 'benign_99_5th_percentile'}

    scores = np.concatenate([benign_scores, attack_scores])
    true_labels = np.concatenate([np.zeros(len(benign_scores), dtype=int), np.ones(len(attack_scores), dtype=int)])
    candidates = np.unique(np.quantile(scores, np.linspace(0.60, 0.999, 120)))

    best_threshold = None
    best_metrics = None
    best_score = -1.0
    for threshold in candidates:
        pred = (scores >= threshold).astype(int)
        precision = precision_score(true_labels, pred, zero_division=0)
        recall = recall_score(true_labels, pred, zero_division=0)
        f1 = f1_score(true_labels, pred, zero_division=0)
        score = (0.7 * f1) + (0.3 * precision)
        if score > best_score:
            best_score = score
            best_threshold = float(threshold)
            best_metrics = {
                'precision': float(precision),
                'recall': float(recall),
                'f1': float(f1),
                'selection_score': float(score),
                'strategy': 'maximize_weighted_f1_precision_on_calibration'
            }
    return best_threshold, best_metrics

def build_autoencoder(input_dim):
    regularizer = keras.regularizers.l2(1e-5)
    inputs = keras.Input(shape=(input_dim,))
    x = keras.layers.GaussianNoise(0.03)(inputs)
    x = keras.layers.Dense(96, kernel_regularizer=regularizer)(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation('relu')(x)
    x = keras.layers.Dropout(0.10)(x)
    x = keras.layers.Dense(48, kernel_regularizer=regularizer)(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation('relu')(x)
    latent = keras.layers.Dense(24, activation='relu', kernel_regularizer=regularizer)(x)
    x = keras.layers.Dense(48, kernel_regularizer=regularizer)(latent)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation('relu')(x)
    x = keras.layers.Dropout(0.05)(x)
    x = keras.layers.Dense(96, kernel_regularizer=regularizer)(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation('relu')(x)
    outputs = keras.layers.Dense(input_dim, activation='linear', dtype='float32')(x)
    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=keras.optimizers.Adam(5e-4), loss=keras.losses.Huber())
    return model

def save_class_distribution_plot(output_dir, y):
    counts = y.value_counts().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(counts.index.astype(str), counts.values, color='#2a6f97')
    ax.set_title('Dataset Class Distribution')
    ax.set_ylabel('Rows')
    ax.set_xlabel('Class')
    ax.tick_params(axis='x', rotation=45)
    fig.tight_layout()
    fig.savefig(os.path.join(output_dir, 'dataset_class_distribution.png'), dpi=160)
    plt.close(fig)

def save_classifier_plots(output_dir, y_test, y_pred, class_names, report):
    cm = confusion_matrix(y_test, y_pred, labels=class_names)
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_title('Random Forest Confusion Matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_xticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha='right')
    ax.set_yticks(range(len(class_names)))
    ax.set_yticklabels(class_names)
    max_value = cm.max() if cm.size else 0
    for row_idx in range(cm.shape[0]):
        for col_idx in range(cm.shape[1]):
            cell = cm[row_idx, col_idx]
            color = 'white' if cell > (max_value / 2 if max_value else 0) else 'black'
            ax.text(col_idx, row_idx, str(cell), ha='center', va='center', color=color, fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(os.path.join(output_dir, 'classifier_confusion_matrix.png'), dpi=160)
    plt.close(fig)

    labels = []
    precision_values = []
    recall_values = []
    f1_values = []
    for class_name in class_names:
        if class_name not in report:
            continue
        labels.append(class_name)
        precision_values.append(report[class_name]['precision'])
        recall_values.append(report[class_name]['recall'])
        f1_values.append(report[class_name]['f1-score'])

    x_positions = np.arange(len(labels))
    width = 0.25
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(x_positions - width, precision_values, width=width, label='Precision', color='#4c956c')
    ax.bar(x_positions, recall_values, width=width, label='Recall', color='#f4a259')
    ax.bar(x_positions + width, f1_values, width=width, label='F1-score', color='#5b8e7d')
    ax.set_title('Random Forest Per-Class Scores')
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.05)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(output_dir, 'classifier_per_class_scores.png'), dpi=160)
    plt.close(fig)

def save_autoencoder_plots(output_dir, history, benign_mse, attack_mse, threshold):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(history.history.get('loss', []), label='Train Loss', color='#1d3557')
    ax.plot(history.history.get('val_loss', []), label='Validation Loss', color='#e76f51')
    ax.set_title('Autoencoder Training Loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(output_dir, 'autoencoder_loss_curve.png'), dpi=160)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(benign_mse, bins=40, alpha=0.7, label='Benign', color='#457b9d')
    if len(attack_mse):
        ax.hist(attack_mse, bins=40, alpha=0.6, label='Attack', color='#d62828')
    ax.axvline(threshold, color='black', linestyle='--', linewidth=1.5, label=f'Threshold={threshold:.4f}')
    ax.set_title('Autoencoder Reconstruction Error Distribution')
    ax.set_xlabel('Reconstruction MSE')
    ax.set_ylabel('Flow Count')
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(output_dir, 'autoencoder_reconstruction_error.png'), dpi=160)
    plt.close(fig)


In [ ]:
download_kaggle_dataset(DATASET_SLUG, DOWNLOAD_DIR)

X, y, detected_label_col = load_balanced_training_data(
    DOWNLOAD_DIR,
    label_override=LABEL_COLUMN_OVERRIDE,
    chunk_size=CHUNK_SIZE,
)
label_col = LABEL_COLUMN_OVERRIDE or detected_label_col

gc.collect()

print('Rows:', len(X))
print('Features:', len(X.columns))
print('Label column:', label_col)
print('Approx X memory (MB):', round(dataframe_mb(X), 2))
print('Classes:', sorted(y.unique().tolist()))


In [ ]:
class_counts = y.value_counts()
stratify_labels = y if (class_counts >= 2).all() else None
if stratify_labels is None:
    print('Warning: some classes have <2 rows after balancing, split will run without stratify')

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=stratify_labels,
)

del X, y
gc.collect()

X_train_rf, y_train_rf, classifier_counts_before, classifier_counts_after = oversample_classifier_training_data(
    X_train,
    y_train,
    CLASSIFIER_MIN_ROWS_PER_CLASS,
    CLASSIFIER_MAX_ROWS_PER_CLASS,
    random_state=RANDOM_STATE,
)

print('Classifier class counts before oversampling:')
print(classifier_counts_before)
print('Classifier class counts after oversampling:')
print(classifier_counts_after)

rf_train_started = time.perf_counter()
rf_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('rf', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])

rf_param_grid = {
    'rf__n_estimators': [300, 400, 500],
    'rf__max_depth': [None, 16, 24, 32],
    'rf__min_samples_split': [2, 5, 8],
    'rf__min_samples_leaf': [1, 2, 3],
    'rf__max_features': ['sqrt', 'log2', 0.5],
    'rf__class_weight': ['balanced', 'balanced_subsample'],
}

X_tune, y_tune = make_rf_tuning_subset(
    X_train,
    y_train,
    RF_TUNE_SAMPLE_MAX_ROWS,
    random_state=RANDOM_STATE,
)
tune_cv_splits = safe_cv_splits(y_tune, RF_CV_SPLITS)
rf_best_params = None
rf_best_cv_macro_f1 = None

if tune_cv_splits >= 2 and y_tune.nunique() >= 2:
    cv = StratifiedKFold(n_splits=tune_cv_splits, shuffle=True, random_state=RANDOM_STATE)
    rf_search = RandomizedSearchCV(
        estimator=rf_pipeline,
        param_distributions=rf_param_grid,
        n_iter=RF_SEARCH_ITER,
        scoring='f1_macro',
        n_jobs=-1,
        cv=cv,
        random_state=RANDOM_STATE,
        verbose=1,
    )

    print(f'Tuning Random Forest on {len(X_tune)} rows with {tune_cv_splits}-fold CV...')
    rf_search.fit(X_tune, y_tune)
    rf_best_params = rf_search.best_params_
    rf_best_cv_macro_f1 = float(rf_search.best_score_)
    rf = rf_search.best_estimator_
else:
    print('Skipping RF hyperparameter search because the tuning subset is too small for stratified CV.')
    rf = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('rf', RandomForestClassifier(
            n_estimators=400,
            class_weight='balanced_subsample',
            max_features='sqrt',
            max_depth=None,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ))
    ])
    rf_best_params = {
        'rf__n_estimators': 400,
        'rf__class_weight': 'balanced_subsample',
        'rf__max_features': 'sqrt',
        'rf__max_depth': None,
    }

print(f'Refitting Random Forest on oversampled train set ({len(X_train_rf)} rows)...')
rf.fit(X_train_rf, y_train_rf)
rf_train_seconds = time.perf_counter() - rf_train_started

y_pred = rf.predict(X_test)
rf_accuracy = accuracy_score(y_test, y_pred)
rf_macro_f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
rf_weighted_f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
rf_report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

print('RF tuning rows:', len(X_tune))
print('RF best params:', rf_best_params)
print('RF best CV macro F1:', rf_best_cv_macro_f1)
print('RF test accuracy:', rf_accuracy)
print('RF test macro F1:', rf_macro_f1)
print('RF test weighted F1:', rf_weighted_f1)
print('RF training seconds:', round(rf_train_seconds, 2))


In [ ]:
X_benign_train_full = X_train.loc[y_train == BENIGN_LABEL].copy()
X_attack_train_full = X_train.loc[y_train != BENIGN_LABEL].copy()
X_benign_test = X_test.loc[y_test == BENIGN_LABEL].copy()
X_attack_test = X_test.loc[y_test != BENIGN_LABEL].copy()

if X_benign_train_full.empty:
    raise ValueError(f'Could not find benign train samples with label {BENIGN_LABEL}')
if X_benign_test.empty:
    raise ValueError(f'Could not find benign test samples with label {BENIGN_LABEL}')

ae_clip_lower, ae_clip_upper = compute_clip_bounds(
    X_benign_train_full,
    lower_q=AE_CLIP_LOWER_QUANTILE,
    upper_q=AE_CLIP_UPPER_QUANTILE,
)

X_benign_train_filtered, ae_filter_metrics = filter_benign_rows_for_autoencoder(
    X_benign_train_full,
    ae_clip_lower,
    ae_clip_upper,
    keep_quantile=AE_KEEP_BENIGN_ROW_QUANTILE,
)

if len(X_benign_train_filtered) < max(2000, int(0.5 * len(X_benign_train_full))):
    print('Filtered benign training rows became too small, fallback to unfiltered benign train set.')
    X_benign_train_filtered = X_benign_train_full.copy()
    ae_filter_metrics['fallback_used'] = True
else:
    ae_filter_metrics['fallback_used'] = False

if len(X_benign_train_filtered) > AE_MAX_BENIGN_TRAIN_ROWS:
    X_benign_train_filtered = X_benign_train_filtered.sample(n=AE_MAX_BENIGN_TRAIN_ROWS, random_state=RANDOM_STATE)

X_benign_fit, X_benign_calib = train_test_split(
    X_benign_train_filtered,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

X_attack_calib = X_attack_train_full.copy()
if len(X_attack_calib) > AE_MAX_ATTACK_CALIB_ROWS:
    X_attack_calib = X_attack_calib.sample(n=AE_MAX_ATTACK_CALIB_ROWS, random_state=RANDOM_STATE)

X_benign_fit_clipped = clip_frame(X_benign_fit, ae_clip_lower, ae_clip_upper)
X_benign_calib_clipped = clip_frame(X_benign_calib, ae_clip_lower, ae_clip_upper)
X_benign_test_clipped = clip_frame(X_benign_test, ae_clip_lower, ae_clip_upper)
X_attack_calib_clipped = clip_frame(X_attack_calib, ae_clip_lower, ae_clip_upper) if not X_attack_calib.empty else X_attack_calib
X_attack_test_clipped = clip_frame(X_attack_test, ae_clip_lower, ae_clip_upper) if not X_attack_test.empty else X_attack_test

ae_scaler = RobustScaler(quantile_range=(5, 95))
X_benign_fit_scaled = ae_scaler.fit_transform(X_benign_fit_clipped).astype(np.float32)
X_benign_calib_scaled = ae_scaler.transform(X_benign_calib_clipped).astype(np.float32)
X_benign_test_scaled = ae_scaler.transform(X_benign_test_clipped).astype(np.float32)
X_attack_test_scaled = ae_scaler.transform(X_attack_test_clipped).astype(np.float32) if not X_attack_test.empty else np.empty((0, X_benign_test.shape[1]), dtype=np.float32)
X_attack_calib_scaled = ae_scaler.transform(X_attack_calib_clipped).astype(np.float32) if not X_attack_calib.empty else np.empty((0, X_benign_fit.shape[1]), dtype=np.float32)

X_b_train, X_b_val = train_test_split(
    X_benign_fit_scaled,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

effective_ae_batch_size = AE_BATCH_SIZE
predict_batch_size = max(AE_BATCH_SIZE * 4, 1024) if GPU_STATUS['gpu_count'] else max(AE_BATCH_SIZE * 2, 512)

train_ds = tf.data.Dataset.from_tensor_slices((X_b_train, X_b_train))
train_ds = train_ds.shuffle(min(len(X_b_train), 10000), seed=RANDOM_STATE, reshuffle_each_iteration=True)
train_ds = train_ds.batch(effective_ae_batch_size).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_b_val, X_b_val))
val_ds = val_ds.batch(effective_ae_batch_size).prefetch(tf.data.AUTOTUNE)

ae_train_started = time.perf_counter()
ae_model = build_autoencoder(X_benign_fit_scaled.shape[1])
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5, verbose=1),
]

print('Training Autoencoder...')
print('Autoencoder GPU status:', GPU_STATUS)
print('AE benign filtering metrics:', ae_filter_metrics)
history = ae_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=AE_EPOCHS,
    verbose=2,
    callbacks=callbacks,
)
ae_train_seconds = time.perf_counter() - ae_train_started

def reconstruction_mse(model, values):
    if len(values) == 0:
        return np.array([])
    recon = model.predict(values, batch_size=predict_batch_size, verbose=0)
    return np.mean(np.square(recon - values), axis=1)

benign_calib_mse = reconstruction_mse(ae_model, X_benign_calib_scaled)
attack_calib_mse = reconstruction_mse(ae_model, X_attack_calib_scaled)
threshold, ae_calibration_metrics = choose_autoencoder_threshold(benign_calib_mse, attack_calib_mse)

benign_mse = reconstruction_mse(ae_model, X_benign_test_scaled)
attack_mse = reconstruction_mse(ae_model, X_attack_test_scaled)

test_scores = np.concatenate([benign_mse, attack_mse])
test_true = np.concatenate([
    np.zeros(len(benign_mse), dtype=int),
    np.ones(len(attack_mse), dtype=int),
])
test_pred = (test_scores >= threshold).astype(int)

ae_test_metrics = {
    'threshold': float(threshold),
    'attack_precision': float(precision_score(test_true, test_pred, zero_division=0)),
    'attack_recall': float(recall_score(test_true, test_pred, zero_division=0)),
    'attack_f1': float(f1_score(test_true, test_pred, zero_division=0)),
    'benign_reconstruction_mse_mean': float(np.mean(benign_mse)),
    'benign_reconstruction_mse_std': float(np.std(benign_mse)),
    'attack_reconstruction_mse_mean': float(np.mean(attack_mse)) if len(attack_mse) else None,
    'attack_reconstruction_mse_std': float(np.std(attack_mse)) if len(attack_mse) else None,
    'separation_ratio_attack_over_benign': float((np.mean(attack_mse) / max(np.mean(benign_mse), 1e-8))) if len(attack_mse) else None,
}

print('AE calibration metrics:', ae_calibration_metrics)
print('AE test metrics:', ae_test_metrics)
print('AE training seconds:', round(ae_train_seconds, 2))


In [ ]:
lime_sample = min(LIME_MAX_TRAIN_ROWS, len(X_train))
lime_frame = X_train.sample(n=lime_sample, random_state=RANDOM_STATE) if len(X_train) > lime_sample else X_train

print('Building LIME explainer...')
explainer = lime_tabular.LimeTabularExplainer(
    training_data=lime_frame.values,
    feature_names=APP_FEATURES,
    class_names=[str(label) for label in sorted(y_train.unique())],
    mode='classification',
    discretize_continuous=True,
    random_state=RANDOM_STATE,
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(os.path.join(OUTPUT_DIR, 'model.pkl'), 'wb') as f:
    pickle.dump(rf, f)

joblib.dump(ae_scaler, os.path.join(OUTPUT_DIR, 'preprocess_pipeline_AE_39ft.save'))
ae_model.save(os.path.join(OUTPUT_DIR, 'autoencoder_39ft.hdf5'))

with open(os.path.join(OUTPUT_DIR, 'explainer'), 'wb') as f:
    dill.dump(explainer, f)

all_labels = pd.concat([y_train, y_test], ignore_index=True)
class_names = sorted(all_labels.astype(str).unique().tolist())
rf_best_cv_macro_f1_text = f'{rf_best_cv_macro_f1:.4f}' if rf_best_cv_macro_f1 is not None else 'n/a'
metrics = {
    'dataset_slug': DATASET_SLUG,
    'label_column': label_col,
    'benign_label': BENIGN_LABEL,
    'feature_order': APP_FEATURES,
    'class_names': class_names,
    'data_balance': {
        'max_benign_rows_total': MAX_BENIGN_ROWS_TOTAL,
        'max_attack_rows_per_class': MAX_ATTACK_ROWS_PER_CLASS,
        'chunk_size': CHUNK_SIZE,
    },
    'classifier_sampling': {
        'min_rows_per_class': CLASSIFIER_MIN_ROWS_PER_CLASS,
        'max_rows_per_class': CLASSIFIER_MAX_ROWS_PER_CLASS,
        'counts_before': classifier_counts_before.to_dict(),
        'counts_after': classifier_counts_after.to_dict(),
    },
    'speed_optimizations': {
        'rf_tune_sample_max_rows': RF_TUNE_SAMPLE_MAX_ROWS,
        'rf_search_iter': RF_SEARCH_ITER,
        'rf_cv_splits': RF_CV_SPLITS,
        'gpu_mixed_precision_enabled': ENABLE_GPU_MIXED_PRECISION,
        'gpu_status': GPU_STATUS,
    },
    'classifier': {
        'best_params': rf_best_params,
        'best_cv_macro_f1': rf_best_cv_macro_f1,
        'tuning_rows': int(len(X_tune)),
        'train_rows_original': int(len(X_train)),
        'train_rows_effective': int(len(X_train_rf)),
        'training_seconds': float(rf_train_seconds),
        'accuracy': float(rf_accuracy),
        'macro_f1': float(rf_macro_f1),
        'weighted_f1': float(rf_weighted_f1),
        'classification_report': rf_report,
    },
    'autoencoder_preprocessing': {
        'clip_lower_quantile': AE_CLIP_LOWER_QUANTILE,
        'clip_upper_quantile': AE_CLIP_UPPER_QUANTILE,
        'keep_benign_row_quantile': AE_KEEP_BENIGN_ROW_QUANTILE,
        'benign_filter_metrics': ae_filter_metrics,
    },
    'autoencoder': {
        'calibration': ae_calibration_metrics,
        'test': ae_test_metrics,
        'epochs_trained': len(history.history['loss']),
        'batch_size': int(effective_ae_batch_size),
        'training_seconds': float(ae_train_seconds),
        'final_train_loss': float(history.history['loss'][-1]),
        'final_val_loss': float(history.history['val_loss'][-1]),
    },
}

with open(os.path.join(OUTPUT_DIR, 'training_metrics.json'), 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

report_lines = [
    '# Training Report V3',
    '',
    f"- Label column: `{label_col}`",
    f"- Benign label: `{BENIGN_LABEL}`",
    f"- Max benign rows: `{MAX_BENIGN_ROWS_TOTAL}`",
    f"- Max attack rows per class: `{MAX_ATTACK_ROWS_PER_CLASS}`",
    f"- Classifier min rows per class after oversampling: `{CLASSIFIER_MIN_ROWS_PER_CLASS}`",
    f"- RF tuning rows: `{len(X_tune)}`",
    f"- RF best CV macro F1: `{rf_best_cv_macro_f1_text}`",
    f"- RF test accuracy: `{float(rf_accuracy):.4f}`",
    f"- RF test macro F1: `{float(rf_macro_f1):.4f}`",
    f"- RF test weighted F1: `{float(rf_weighted_f1):.4f}`",
    f"- RF training seconds: `{float(rf_train_seconds):.2f}`",
    f"- AE threshold: `{ae_test_metrics['threshold']:.6f}`",
    f"- AE attack precision: `{ae_test_metrics['attack_precision']:.4f}`",
    f"- AE attack recall: `{ae_test_metrics['attack_recall']:.4f}`",
    f"- AE attack F1: `{ae_test_metrics['attack_f1']:.4f}`",
    f"- AE separation ratio attack/benign: `{ae_test_metrics['separation_ratio_attack_over_benign']}`",
    f"- AE training seconds: `{float(ae_train_seconds):.2f}`",
    f"- GPU count: `{GPU_STATUS['gpu_count']}`",
    f"- Mixed precision policy: `{GPU_STATUS['mixed_precision_policy']}`",
]

with open(os.path.join(OUTPUT_DIR, 'training_report.md'), 'w', encoding='utf-8') as f:
    f.write('\n'.join(report_lines))

save_class_distribution_plot(OUTPUT_DIR, all_labels)
save_classifier_plots(OUTPUT_DIR, y_test, y_pred, class_names, rf_report)
save_autoencoder_plots(OUTPUT_DIR, history, benign_mse, attack_mse, ae_test_metrics['threshold'])

zip_path = shutil.make_archive(OUTPUT_DIR, 'zip', OUTPUT_DIR)
print('Saved artifacts to', OUTPUT_DIR)
print('Zip file:', zip_path)
print('Output files:', sorted(os.listdir(OUTPUT_DIR)))


In [ ]:
metrics = json.loads(Path(OUTPUT_DIR, 'training_metrics.json').read_text(encoding='utf-8'))
print(json.dumps({
    'label_column': metrics['label_column'],
    'classifier_best_cv_macro_f1': metrics['classifier']['best_cv_macro_f1'],
    'classifier_accuracy': metrics['classifier']['accuracy'],
    'classifier_macro_f1': metrics['classifier']['macro_f1'],
    'classifier_weighted_f1': metrics['classifier']['weighted_f1'],
    'ae_attack_precision': metrics['autoencoder']['test']['attack_precision'],
    'ae_attack_recall': metrics['autoencoder']['test']['attack_recall'],
    'ae_attack_f1': metrics['autoencoder']['test']['attack_f1'],
    'ae_separation_ratio': metrics['autoencoder']['test']['separation_ratio_attack_over_benign'],
    'ae_threshold': metrics['autoencoder']['test']['threshold'],
}, indent=2, ensure_ascii=False))


In [ ]:
plot_files = [
    'dataset_class_distribution.png',
    'classifier_confusion_matrix.png',
    'classifier_per_class_scores.png',
    'autoencoder_loss_curve.png',
    'autoencoder_reconstruction_error.png',
]

for plot_name in plot_files:
    img = mpimg.imread(Path(OUTPUT_DIR, plot_name))
    plt.figure(figsize=(10, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title(plot_name)
    plt.show()


In [ ]:
zip_path = Path(f'{OUTPUT_DIR}.zip')
assert zip_path.exists(), f'Could not find {zip_path}'
files.download(str(zip_path))
